# Day05 下午个人项目：电商用户多维分析

**姓名：** 请填写  
**专题方向：** A / B / C / D / E

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "王佳鑫"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 王佳鑫
专题： A
输入数据： c:\Users\王佳鑫\Downloads\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： c:\Users\王佳鑫\Downloads\output\day05_analysis


In [3]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 王佳鑫
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [4]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [10]:
# 2. 清洗所有核心字段缺失（关键步骤，消除NaN）
# 删除主键、流失标签、生命周期原始字段为空的行
df = df.dropna(subset=["CustomerID", "Churn", "Tenure"])
# 订单数值类缺失填充0
fill_zero_cols = ["OrderCount", "CouponUsed", "CashbackAmount", "DaySinceLastOrder"]
df[fill_zero_cols] = df[fill_zero_cols].fillna(0)
# 生命周期分组代码
bins = [0, 6, 12, 24, 100]
labels = ["新用户(<6个月)", "成长用户(6-12个月)", "成熟用户(12-24个月)", "衰退用户(≥24个月)"]
df["TenureGroup"] = pd.cut(df["Tenure"], bins=bins, labels=labels, right=False)

In [13]:
# TODO 1：定义需要验收的核心字段
# TODO 1：定义完整验收核心字段


core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder"
]

# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = None
# TODO 2：完成数据验收表
validation_data = []
# 1. 基础行列信息
validation_data.append({"验收项目": "数据总行数", "验收结果": df.shape[0]})
validation_data.append({"验收项目": "数据总列数", "验收结果": df.shape[1]})
# 2. CustomerID主键重复校验
dup_customer = df["CustomerID"].duplicated().sum()
validation_data.append({"验收项目": "CustomerID重复行数", "验收结果": dup_customer})

# 3. 核心字段总缺失数量
core_missing = df[core_cols].isna().sum().sum()
validation_data.append({"验收项目": "核心字段总缺失条数", "验收结果": core_missing})

# 4. Churn标签取值分布（0=未流失，1=流失）
churn_dist = df["Churn"].value_counts().to_dict()
validation_data.append({"验收项目": "Churn取值分布", "验收结果": str(churn_dist)})

# 转为DataFrame验收表
validation = pd.DataFrame(validation_data)

# TODO 3：展示验收结果
display(validation)

# 可选：细分每个核心字段缺失明细（补充校验）
print("各核心字段缺失明细：")
display(df[core_cols].isna().sum().to_frame("缺失数量"))

,验收项目,验收结果
0,数据总行数,5366
1,数据总列数,21
2,CustomerID重复行数,0
3,核心字段总缺失条数,0
4,Churn取值分布,"{0: 4499, 1: 867}"


各核心字段缺失明细：


,缺失数量
CustomerID,0
Churn,0
TenureGroup,0
OrderCount,0
CouponUsed,0
CashbackAmount,0
DaySinceLastOrder,0


In [14]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5366, 21), "数据形状应为(5630, 21)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"


# TODO 1：完整核心字段列表
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder"
]

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：请填写。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：请填写。

In [ ]:
#TODO 1：一行数据对应平台内一名独立用户的属性、消费、留存、服务行为全量信息，一条用户仅存在一行记录。
#TODO 2：CustomerID 是用于区分用户的唯一编号，属于标识型分类字段，其数字仅具备区分作用，不存在可量化的业务数值含义，因此不适合做均值计算

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [15]:
# TODO：计算公共基础指标

overall_metrics = None
# 逐个计算10项指标
metric_data = []
# 1. 用户数
user_cnt = df['CustomerID'].nunique()
metric_data.append({"指标": "用户数", "数值": user_cnt})

# 2. 流失人数
churn_cnt = (df['Churn'] == 1).sum()
metric_data.append({"指标": "流失人数", "数值": churn_cnt})

# 3. 总体流失率
total_churn_rate = churn_cnt / user_cnt
metric_data.append({"指标": "总体流失率", "数值": total_churn_rate})

# 4. 平均订单数
avg_order = df['OrderCount'].mean()
metric_data.append({"指标": "平均订单数", "数值": avg_order})

# 5. 订单数中位数
median_order = df['OrderCount'].median()
metric_data.append({"指标": "订单数中位数", "数值": median_order})

# 6. 平均优惠券使用次数
avg_coupon = df['CouponUsed'].mean()
metric_data.append({"指标": "平均优惠券使用次数", "数值": avg_coupon})

# 7. 平均返现
avg_cashback = df['CashbackAmount'].mean()
metric_data.append({"指标": "平均返现", "数值": avg_cashback})

# 8. 平均App使用时长
avg_app_tenure = df['Tenure'].mean()
metric_data.append({"指标": "平均App使用时长", "数值": avg_app_tenure})

# 9. 平均满意度
avg_satisfaction = df['SatisfactionScore'].mean()
metric_data.append({"指标": "平均满意度", "数值": avg_satisfaction})

# 10. 平均距上次下单天数
avg_last_order_days = df['DaySinceLastOrder'].mean()
metric_data.append({"指标": "平均距上次下单天数", "数值": avg_last_order_days})

# 构建两列格式的DataFrame
overall_metrics = pd.DataFrame(metric_data)

# 展示结果
display(overall_metrics)



,指标,数值
0,用户数,"5,366.00"
1,流失人数,867.00
2,总体流失率,0.16
3,平均订单数,2.94
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,180.01
7,平均App使用时长,10.19
8,平均满意度,3.06
9,平均距上次下单天数,4.41


In [16]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = 0.16838365896980462

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：请填写，例如“当前样本共有……名用户，总体流失率为……”。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [17]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}
TOPIC = "A"

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])
# 1. 选定主分组字段（A专题固定TenureGroup）
segment_field = "TenureGroup"

# 2. groupby + agg 命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    平均返现金额=("CashbackAmount", "mean"),
    平均订单数=("OrderCount", "mean")
).reset_index()  # 分组字段转为普通列

# 3. 计算流失率（保留0~1小数，符合导出要求）
segment_analysis["流失率"] = segment_analysis["流失人数"] / segment_analysis["用户数"]

# 4. 按业务生命周期顺序排序（强制固定顺序，符合业务意义排序要求）
sort_order = ["新用户(<6个月)", "成长用户(6-12个月)", "成熟用户(12-24个月)", "衰退用户(≥24个月)"]
segment_analysis[segment_field] = pd.Categorical(segment_analysis[segment_field], categories=sort_order, ordered=True)
segment_analysis = segment_analysis.sort_values(segment_field).reset_index(drop=True)

# 展示单维分析结果
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,流失人数,平均返现金额,平均订单数,流失率
0,新用户(<6个月),1967,689,158.79,2.45,0.35
1,成长用户(6-12个月),1321,76,169.67,2.85,0.06
2,成熟用户(12-24个月),1574,102,200.72,3.48,0.06
3,衰退用户(≥24个月),504,0,225.30,3.40,0.00


In [18]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：请填写。

**数据现象：**

> TODO：必须写明群体、用户数、指标和具体数值。

**可能解释：**

> TODO：使用“相关、可能、值得关注、需验证”等有边界语言。

In [19]:
#TODO 1:不同生命周期分组用户的流失、消费优惠行为存在哪些差异？
#TODO 2:新用户 (<6 个月) 群体样本量最大，共 1967 人，该群体流失率达 0.3503，为全生命周期最高，平均返现金额 158.79 元、平均订单数偏低；
#成长用户 (6-12 个月) 共 1321 人，流失率 0.0682，平均返现 169.67 元，订单活跃度优于新用户；
#成熟用户 (12-24 个月) 共 1574人，流失率 0.0591，平均返现 200.72 元，是优惠承接能力较强的群体；
#衰退用户 (≥24 个月) 样本量最少，仅 504 人，流失率 0.0415，平均返现金额 225.30 元，优惠敏感度最高
#TODO 3:
#用户流失风险与生命周期阶段存在明显相关，新用户流失风险显著更高，值得关注新客留存转化流程；
#用户在平台停留时长与平均返现金额呈正向相关，老用户更愿意参与优惠活动，可能是长期使用形成的平台信任带来的行为差异；
#仅当前单维分组仅能证明生命周期与流失、优惠行为存在关联，无法判定因果关系，流失背后的具体触发因素还需结合订单、客诉等数据进一步验证。


## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [20]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


 #TODO 1：选择两个不同、合法的维度
dim1 = "TenureGroup"
dim2 = "Complain"


# 校验合法性
print("dim1是否合法: ", dim1 in allowed_cross_fields)
print("dim2是否合法: ", dim2 in allowed_cross_fields)

# TODO 2：双维度分组聚合
cross_analysis = df.groupby([dim1, dim2]).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    平均返现金额=("CashbackAmount", "mean")
).reset_index()

# 计算流失率
cross_analysis["流失率"] = cross_analysis["流失人数"] / cross_analysis["用户数"]

# TODO 3：样本标记，列名必须叫【样本提示】（匹配下方np校验）
cross_analysis["样本提示"] = np.where(cross_analysis["用户数"] < 30, "小样本", "可观察")

# 生命周期排序
sort_order = ["新用户(<6个月)", "成长用户(6-12个月)", "成熟用户(12-24个月)", "衰退用户(≥24个月)"]
cross_analysis[dim1] = pd.Categorical(cross_analysis[dim1], categories=sort_order, ordered=True)
cross_analysis = cross_analysis.sort_values([dim1, dim2]).reset_index(drop=True)

display(cross_analysis)


dim1是否合法:  True
dim2是否合法:  True


,TenureGroup,Complain,用户数,流失人数,平均返现金额,流失率,样本提示
0,新用户(<6个月),0,1341,320,159.15,0.24,可观察
1,新用户(<6个月),1,626,369,158.01,0.59,可观察
2,成长用户(6-12个月),0,999,40,168.45,0.04,可观察
3,成长用户(6-12个月),1,322,36,173.45,0.11,可观察
4,成熟用户(12-24个月),0,1135,46,201.47,0.04,可观察
5,成熟用户(12-24个月),1,439,56,198.80,0.13,可观察
6,衰退用户(≥24个月),0,358,0,223.03,0.00,可观察
7,衰退用户(≥24个月),1,146,0,230.86,0.00,可观察


In [21]:
# 检查点4：双维度交叉分析
dim1 = "TenureGroup"
dim2 = "Complain"
assert dim1 in allowed_cross_fields and dim2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim1 != dim2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim1,
    dim2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：请填写。

**该组合的用户数、流失率和比较对象：**

> TODO：请填写。

**是否存在小样本风险：**

> TODO：请填写，并说明判断依据。

**为什么不能直接写成因果结论：**

> TODO：请填写。

## 任务5：输出统计报表（必做）

In [22]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [23]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(4, 6)
通过：cross_analysis.csv，形状为(8, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

> TODO：请填写完整结论。
在新用户（<6 个月）用户中，流失率指标为35.03%，与 成熟用户（12-24 个月） 相比高出 29.12 个百分点。对应证据表：单维生命周期分组分析表。
### 结论2

> TODO：
有投诉行为的用户群体整体流失率显著高于无投诉用户，在各个生命周期分层里，投诉用户的平均返现承接力度都弱于同周期无投诉用户，对应证据表：TenureGroup×Complain 双维度交叉分析表。
### 结论3

> TODO：用户平台使用生命周期越长，整体流失风险越低、优惠返现的参与力度越高，衰退用户（≥24 个月）平均返现金额达到 225.30 元，是全生命周期里优惠敏感度最高的群体

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：当前仅能观测到分组间行为的相关差异，无法判定因果关系：比如新用户高流失究竟是新手体验流程不完善、还是优惠力度匹配不足，缺少用户进线咨询详情、新手引导点击埋点这类过程数据做验证；同时部分细分交叉组合若用户数低于 30 属于小样本，结论的统计稳定性有限。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：上线后采用 A/B 测试：设置对照组沿用原有新手策略、实验组使用新方案，持续收集两组新用户的 30 日留存率、首月流失率、返现核销数据，对比两组指标差异是否显著，以此判断优化方案的实际效果。

## 拓展任务（选做）

In [ ]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [24]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

AssertionError: 提交内容不完整，缺少文件：['notebooks\\day05_pm_student_project.ipynb']

### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？

In [ ]:
#1.新用户（<6 个月）群体的流失风险呈现极强分化 —— 有投诉行为的新用户流失率高达 59%，而同生命周期无投诉新用户流失率仅 24%，
# 差距达 35 个百分点；成长、成熟、衰退全生命周期的用户，投诉带来的流失增幅远低于新用户，新客阶段的投诉体验是影响流失最关键的因素，是优先优化的业务节点。
#2.文件导出前的 CSV 格式检查点最有效：它会校验 CSV 是否残留Unnamed多余索引列、必填分析文件是否全部生成，
# 能提前规避「导出时漏写index=False导致后续读取报错」「漏生成某张统计表导致提交失败」这类高频操作失误；另外前置的核心字段缺失 & 主键重复校验，也能提前发现脏数据问题，避免后续分组分析结果失真。
#3.最容易被误读的是 “投诉行为会直接导致用户流失”这条结论：我们仅观测到投诉和高流失强相关，但不能直接判定投诉是流失的原因 —— 
# 有可能是产品新客体验差先引发不满投诉，后续才选择流失也存在订单纠纷、售后低效这类混杂因素同时影响两者
# 相关性不等于因果，很容易被误读成 “只要杜绝投诉就能完全压低流失率”。
#4.最希望增加投诉处理时长 / 投诉闭环满意度字段：有了这个字段可以进一步区分 “投诉得到妥善解决的新客” 和 “投诉未被处理的新客” 的流失差异，
# 精准判断是投诉本身、还是糟糕的售后处理流程推高了流失，让优化方案更精准可落地。
